In [45]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

with open("config/config.json", "r") as f:
    config = json.load(f)

print(config)

{'llm': {'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}, 'evaluation': {'criteria': ['tone_empathy', 'knowledge_accuracy', 'resolution_quality'], 'score_range': [1, 5]}, 'classification': {'labels': ['billing', 'claims', 'complaint', 'general_query']}}


In [46]:
import pandas as pd

df = pd.read_csv("data/transcripts.csv")

df.head(4)

,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [47]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(config):
    provider = os.getenv('LLM_PROVIDER')

    if provider.lower() == "openai":
        llm = ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )
    elif provider.lower() == "gemini":
        llm = ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )
    else:
        raise ValueError("Invalid LLM Provider")
    return llm

llm  = load_llm(config)
print(f"LLM initialized with provider: {os.getenv('LLM_PROVIDER')}")

response = llm.invoke("Say 'Setup Completed' in one short sentence")

print(f"LLM Response: {response.content}")

LLM initialized with provider: openai
LLM Response: Setup completed.


In [48]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Classification(BaseModel):
    call_type: str = Field(description="Type of customer call")
    confidence: float = Field(description= "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object = Classification)

prompt = PromptTemplate(
    template="""
You are a call classification assistant.

Classify the following customer support transcript into one of these categories:
{labels}

Transcript:
{transcript}

{format_instructions}
""",
input_variables=["transcript"],
partial_variables={
    "format_instructions": parser.get_format_instructions(),
    "labels": config["classification"]["labels"]
}
)

classification_chain = prompt | llm | parser

sample_text = df.iloc[0]["transcript"]
result = classification_chain.invoke({
    "transcript": sample_text
})

print(f"Classification Result: \n{result}")

Classification Result: 
call_type='billing' confidence=0.95


In [49]:
from tqdm import tqdm

results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Classifying calls..."):
    try:
        output = classification_chain.invoke({
            "transcript": row["transcript"]
        })

        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": output.call_type,
            "confidence": output.confidence
        })
    except Exception as e:
        print(f"Error at row: {i} : {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

results_df = pd.DataFrame(results)
df = df.merge(results_df, on="call_id")

df.head()

Classifying calls...:   0%|          | 0/4 [00:00<?, ?it/s]

Classifying calls...: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]


,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.90
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.90


In [50]:
# --- Step 1: Define routing logic ---

def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    
    elif call_type == "general_query":
        return ["knowledge_accuracy"]
    
    else:
        return ["knowledge_accuracy"]  # fallback


# --- Step 2: Apply routing to dataset ---

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("✅ Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

✅ Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution_quality]
1        2              claims  [knowledge_accuracy, resolution_quality]
2        3           complaint        [tone_empathy, resolution_quality]
3        4       general_query                      [knowledge_accuracy]


In [51]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# define an output schema for tone evaluation
class ToneEvaluation(BaseModel):
    score: int = Field(description = "Score between 1 and 5")
    reasoning: str = Field(description = "Explanation of the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluation)

tone_prompt = PromptTemplate(
    template = """
    You are a QA evaluator for customer support calls.

    Evaluate the agent's tone and empathy in the following transcript.

    Consider:
    - Did the agent acknowledge the cusotmer's issue?
    - Was the tone polite and professional enough?
    - Did the agent show empathy?

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions()
    }

)

tone_chain = tone_prompt | llm | tone_parser

sample_text = df[df["predicted_call_type"] == "complaint"].iloc[0]["transcript"]

result = tone_chain.invoke({
    "transcript": sample_text
})

print(f"Tone evaluation result: \n{result}")

Tone evaluation result: 
score=3 reasoning="The agent acknowledged the customer's issue by apologizing for the inconvenience, which shows some level of empathy. However, the response was brief and did not fully address the customer's frustration about the repeated calls. The tone was polite and professional, but it could have been more empathetic by expressing understanding of the customer's feelings and offering a more detailed solution."


In [52]:
# --- Step 1: Schema ---
class ResolutionEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

resolution_parser = PydanticOutputParser(pydantic_object=ResolutionEvaluation)

# --- Step 2: Prompt ---
resolution_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the resolution quality of the agent.

Consider:
- Did the agent fully resolve the customer's issue?
- Were next steps clearly communicated?
- Did the agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": resolution_parser.get_format_instructions()
    }
)

# --- Step 3: Chain ---
resolution_chain = resolution_prompt | llm | resolution_parser

# --- Step 4: Test
sample_text = df.iloc[0]["transcript"]

result = resolution_chain.invoke({
    "transcript": sample_text
})

print("✅ Resolution Evaluation Result:")
print(result)

✅ Resolution Evaluation Result:
score=2 reasoning="The agent did not fully resolve the customer's issue as the transcript ends without confirming whether the double charge was addressed or if any corrective action was taken. Additionally, there were no next steps communicated to the customer, and the agent did not confirm resolution before ending the call."


In [ ]:
# --- Step 1: Imports (safe to repeat) ---
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# --- Step 2: Schema ---
class KnowledgeEvaluation(BaseModel):
    score: int = Field(description="Score between 1 and 5")
    reasoning: str = Field(description="Explanation of the score")

knowledge_parser = PydanticOutputParser(pydantic_object=KnowledgeEvaluation)

# --- Step 3: Prompt ---
knowledge_prompt = PromptTemplate(
    template="""
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity.

Consider:
- Did the agent provide correct and relevant information?
- Was the explanation clear and easy to understand?
- Did the agent avoid vague or misleading statements?

IMPORTANT:
- If the transcript does not contain enough information, give a moderate score (2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
    input_variables=["transcript"],
    partial_variables={
        "format_instructions": knowledge_parser.get_format_instructions()
    }
)

# --- Step 4: Chain ---
knowledge_chain = knowledge_prompt | llm | knowledge_parser

sample_text = df.iloc[1]["transcript"]

result = knowledge_chain.invoke({
    "transcript": sample_text
})

print("✅ Knowledge Chain Ready")
print(f"Result: {result}")

✅ Knowledge Chain Ready
Result: score=2 reasoning="The transcript does not provide enough information to evaluate the agent's knowledge accuracy and clarity fully. While the agent begins to engage with the customer, there is no detailed information or explanation provided regarding the coverage details. Therefore, it's difficult to assess the correctness or clarity of the information given."
